# Example notebook make n(z) with KNN

This will run the KNN algorithm to get p(z) for all objects in the WFD sample,
then it will use the mode of the p(z) to get the tomographic bin assignments and stack the p(z) to get n(z)

#### Standard imports and rail catalog setup

In [ ]:
import tables_io, qp
import numpy as np
import os
import matplotlib.pyplot as plt
from rail.estimation.algos.k_nearneigh import KNearNeighEstimator, KNearNeighInformer
from rail.core.data import TableHandle
from rail.utils import catalog_utils

# RAIL setup
catalog_utils.clear()
catalog_utils.load_yaml("../tests/catalogs.yaml")
CATALOG_TAG = "cardinal_roman_rubin"
catalog_utils.apply(CATALOG_TAG)

#### Paths for nz challenge related files

In [ ]:
submit_dir = 'submission/rail_knn_4tasks'
model_dir = '../models/rail_knn_4tasks'
public_dir = '../public'
try:
    os.makedirs(submit_dir)
except:
    pass

In [ ]:
### Binning and gridding, these whould come from nz_data_challenge

In [ ]:
n_bins = 5
bin_edges = np.array([0., 0.32, 0.47, 0.61, 0.78, 2.5 ])
z_min = 0
z_max = 1.5
n_grid_points = 151
grid_edges = np.linspace(z_min, z_max, n_grid_points)
grid_centers = 0.5*(grid_edges[0:-1]+grid_edges[1:])
bin_centers = 0.5*(bin_edges[0:-1]+bin_edges[1:])

In [ ]:
taskset = ['taskset_1'] # 'taskset_2']
sims = ['cardinal', 'flagship']
scenarios = ['1yr', '4yr']

# loop over tasksets, simulations and scenarios
for taskset in taskset:
    for sim in sims:
        for scenario in scenarios:

            # Make the file names for this particular setup
            wfd_file = f"{public_dir}/nz_challenge_{taskset}_{sim}_{scenario}_wfd.hdf5"
            model_file = f"{model_dir}/pz_challenge_{taskset}_{sim}_pz_model_1yr.pkl"
            nz_file = f"{submit_dir}/nz_challenge_{taskset}_{sim}_{scenario}_nz_estimate_wfd.hdf5"
            bhat_file = f"{submit_dir}/nz_challenge_{taskset}_{sim}_{scenario}_bhat_wfd.hdf5"

            # Make a KNN estiamtor and run it
            knn_estimate = KNearNeighEstimator.make_stage(
                name=f"estimate_{taskset}_{sim}_{scenario}",
                model=model_file,
                hdf5_groupname="",
                output_mode="return",
                nzbins=301,
                zmax=3.0,
                chunk_size=10000,
                nondetect_val=np.nan,
            )
            test_handle = TableHandle(f"test_{taskset}_{sim}_{scenario}", path=str(wfd_file))
            output = knn_estimate.estimate(test_handle)

            # Read the input data to get the object_ids
            test_data = tables_io.read(test_handle.path)

            # Get the mode of the p(z) and use it for bin assignments
            z_est = np.squeeze(output.data.ancil['zmode'])
            bin_assignments = np.digitize(z_est, bin_edges[1:-1])

            # Count the objects in each bin
            n_objects = np.bincount(bin_assignments)
            hist_list = []

            # Stack the pdfs and build the output qp Ensemble
            pdfs = output.data.pdf(grid_centers)            
            for i in range(n_bins):
                binx = pdfs[bin_assignments==i]
                binx_normed = binx.sum(axis=0)/binx.sum()
                hist_list.append(binx_normed)
            pdfs = np.vstack(hist_list)
            ens = qp.Ensemble(qp.hist, data=dict(bins=grid_edges, pdfs=pdfs))

            # Attach the number of objects and bin-normalization to the pdfs
            ens.set_ancil(
                dict(
                    n_object=n_objects,
                    bin_norms=pdfs.sum(axis=1),
                )
            )
            # Write the n(z) file
            ens.write_to(nz_file)

            # Create and write the bin assignment (bhat) file
            bhat = dict(
                bhat_for_wide_data=bin_assignments, 
                object_id=test_data['object_id'],
            )
            tables_io.write(bhat, bhat_file)